In [1]:
import os
import glob
import gzip
import subprocess
import pandas as pd

In [1]:
def get_desc_taxa(node, cand_taxa):
    desc_taxa = []
    if node.sci_name in cand_taxa:
        desc_taxa.append(node.sci_name)
        return desc_taxa
    for ch in node.children:
        desc_taxa.extend(get_desc_taxa(ch, cand_taxa))
    return desc_taxa

In [2]:
def write_bs_ids_file(bs_ids_file, bs_ids):
    with open(bs_ids_file, 'w') as outf:
        outf.write('\n'.join(bs_ids))

def read_bs_ids_file(bs_ids_file):
    with open(bs_ids_file, 'r') as inf:
        lines = inf.read().split('\n')
        array_id2bs_id = dict(zip(range(1, len(lines) + 1), lines))
    return array_id2bs_id, {v:k for k, v in array_id2bs_id.items()}

In [3]:
def write_preprocessing_script(script_fn, array_str, node_str):
    runstr="""#!/bin/bash -l
#SBATCH --array=ARRAY_STRING
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --nodelist=NODE_STRING
#SBATCH --mem=100g
#SBATCH --time=6:00:00
#SBATCH --job-name=pp
#SBATCH --output=%x_%A_%a.out
#SBATCH --error=%x_%A_%a.err

modulesld
ebld
module use /software/anaconda3/envs/eb/easybuild/modules/all
ml SRA-Toolkit
ml cutadapt
ml FastQC

# file listing all BeeStrong ids (bs_id) for parallel computing
bs_ids_file=$1
sra_path=$2
fastq_path=$3
sample_SRA_file=$4

# work on scratch node is better when lots of I/O operations
node_scratch=/scratch/${USER}/tmp_${SLURM_ARRAY_JOB_ID}_${SLURM_ARRAY_TASK_ID}/
mkdir -p $node_scratch

# getting bs_id
bs_id=$(sed -n ${SLURM_ARRAY_TASK_ID}'{p;q}' ${bs_ids_file})

echo "1. find all SRA ids corresponding to the bs_id and copy them to the scrach node"
sra_ids=()
for sra in $(grep $bs_id $sample_SRA_file | cut -f 2); do
    sra_ids+=($sra)
    echo $sra_path$sra
    cp -r $sra_path$sra $node_scratch
done

# initiate fastq files
fastq_1=$node_scratch${bs_id}_1.fastq
fastq_2=$node_scratch${bs_id}_2.fastq
touch $fastq_1
touch $fastq_2

echo "2. concatenate fastq from multiple SRA files and clean"
for sra in "${sra_ids[@]}"; do
    ls $node_scratch$sra
    fasterq-dump $node_scratch$sra -O $node_scratch
    cat ${node_scratch}${sra}_1.fastq >> $fastq_1
    cat ${node_scratch}${sra}_2.fastq >> $fastq_2
    rm -f ${node_scratch}${sra}_1.fastq
    rm -f ${node_scratch}${sra}_2.fastq
done

echo "3. trim poly-G tails"
fastq_1_t=$node_scratch${bs_id}_1_trim.fastq
fastq_2_t=$node_scratch${bs_id}_2_trim.fastq

cutadapt -a "G{10}" -A "G{10}" -m 50 -o $fastq_1_t -p $fastq_2_t $fastq_1 $fastq_2

mv -f $fastq_1_t $fastq_1
mv -f $fastq_2_t $fastq_2

echo "4. FastQC"
fastqc $fastq_1 $fastq_2

echo "5. compress"
gzip -f $fastq_1 $fastq_2

echo "6. copy to home and clean"
mv -f $node_scratch${bs_id}* $fastq_path
rm -rf $node_scratch

echo DONE""".replace("ARRAY_STRING", array_str).replace("NODE_STRING", node_str)
    with open(script_fn, 'w') as outf:
        outf.write(runstr)

## Slurm monitoring

In [ ]:
def write_array_str(array_ids_to_run, parallel_job_nr=100):
    '''
    convoluted way to build array string (as BSUB argument cannot be too long)
    '''
    if len(array_ids_to_run) == 1:
        return str(array_ids_to_run[0])
    array_str_list = []
    start = array_ids_to_run[0]
    for i in range(1, len(array_ids_to_run)):
        curr = array_ids_to_run[i]
        prev = array_ids_to_run[i - 1]
        if curr != prev + 1:
            if start == prev:
                array_str_list.append(str(prev))
            else:
                array_str_list.append('{}-{}'.format(start, prev))
            start = curr
    # end 
    if start == prev:
        array_str_list.append(str(prev))
    else:
        array_str_list.append('{}-{}'.format(start, prev + 1)) ## this + 1 is a hack
    start = curr
    
    return '{}%{}'.format(','.join(array_str_list), parallel_job_nr)

def get_status_job_array_ids(status='RUNNING', job_name='pp'):
    command = ['squeue', '-t', status, '-n', job_name, '-o', '%.18i']
    result = subprocess.run(command, capture_output=True, text=True)
    job_array_ids = []
    # to handle pending jobs...
    for job_str in result.stdout.split()[1:]:
        job_str = job_str.split('_')[1]
        if job_str.startswith('['):
            split_job_str = job_str[1:].split('%')[0].split(',')
            for js in split_job_str:
                if len(js.split('-')) > 1:
                    x = js.split('-')
                    job_array_ids.extend(list(range(int(x[0]), int(x[1]) + 1)))
                else:
                    job_array_ids.append(int(js))
        else:
            job_array_ids.append(int(job_str))
    return job_array_ids

def get_jobs_to_restart(step_name, completed_jobs_step_before, array_id2bs_id, step_path, expected_files, job_id="*"):

    # capture completed, running, pending, and failed/to_run jobs
    running_jobs = get_status_job_array_ids(status='RUNNING', job_name=step_name)
    pending_jobs = get_status_job_array_ids(status='PENDING', job_name=step_name)
    completed_jobs = []
    
    for array_id in completed_jobs_step_before:
        bs_id = array_id2bs_id[array_id]
        
        # ensure latest log file is there and finishes by DONE
        x = glob.glob('{}/{}_{}_{}.out'.format(step_path, step_name, job_id, array_id))
        if len(x) != 1:
            if len(x) > 1: 
                print(x)
            continue
    
        out_file = x[0]
        with open(out_file, 'r') as inf:
            lines = inf.readlines()
            no_error = True
            for l in lines:
                if l.startswith('(ERR)'):
                    no_error = False
            done = lines and lines[-1].startswith('DONE')

        # expected files (glob format) 
        expected_files_correct = True
        for exp_file in expected_files:
            fn = exp_file.replace('BSID', bs_id)
            if not os.path.exists(fn) or not os.path.getsize(fn) > 100:
                expected_files_correct = False
                break
        
        if done and no_error and expected_files_correct:
            completed_jobs.append(array_id)
            
    jobs_to_run = list(set(completed_jobs_step_before).difference(completed_jobs + running_jobs + pending_jobs))
    return jobs_to_run, completed_jobs

## Bowtie2

In [1]:
def parse_flagstat(flagstat_fn):
    """
    Parse a samtools flagstat output file.
    Returns number of reads for:
        - total
        - with_itself_and_mate_mapped
        - singletons
    """
    results = {
        "total": None,
        "with_itself_and_mate_mapped": None,
        "singletons": None
    }
    with open(flagstat_fn, "r") as inf:
        for line in inf:
            if "in total" in line:
                results["total"] = int(line.split()[0])
            
            elif "with itself and mate mapped" in line:
                results["with_itself_and_mate_mapped"] = int(line.split()[0])

            elif "singletons" in line:
                results["singletons"] = int(line.split()[0])
                
    return results

def count_lines_gz(gzip_fn):
    ''' faster'''
    with gzip.open(gzip_fn, "rb") as inf:
        return sum(buf.count(b"\n") for buf in iter(lambda: inf.read(1024*1024), b""))

def get_total_read_nr_flagstat(bowtie2_path, bs_id, idx_name):
    return parse_flagstat('{}{}_{}_mapped.flagstat'.format(bowtie2_path, bs_id, idx_name))['total']

def get_nonbee_read_nr_flagstat(bowtie2_path, bs_id, idx_name):
    flagstat_res = parse_flagstat('{}{}_{}_mapped.flagstat'.format(bowtie2_path, bs_id, idx_name))
    return (flagstat_res['total'] - flagstat_res['with_itself_and_mate_mapped'] - 2 * flagstat_res['singletons']) / 2

## kraken

In [2]:
def parse_kreport(file_path, bs_id, krakdb, readpool, mhg, cs, sf, r):
    kreport_fn = '{}{}_{}_{}_mhg{}_cs{}_sf{}_rep{}.k2report'.format(file_path, bs_id, krakdb, readpool, mhg, cs, str(sf).replace('.', ''), r)
    if not os.path.exists(kreport_fn): 
        print('missing {}'.format(kreport_fn))
    with open(kreport_fn, 'r') as inf:
        ucseqs_nr = int(inf.readline().split()[1])
        cseqs_nr = int(inf.readline().split()[1])
    return ucseqs_nr, cseqs_nr

def parse_breport(bracken_path, bs_id, krakdb, readpool, mhg, cs, sf, r, level, taxa):
    '''get read numbers for taxa'''
    breport_fn = '{}{}_{}_{}_mhg{}_cs{}_sf{}_rep{}_{}.breport'.format(bracken_path, bs_id, krakdb, readpool, mhg, cs, str(sf).replace('.', ''), r, level)
    if not os.path.exists(breport_fn): 
        print('missing {}'.format(breport_fn))
    breport_df = pd.read_csv(breport_fn, header=None, sep='\t')
    breport_df[5] = [x.lstrip() for x in breport_df[5].to_list()]
    return breport_df[breport_df[5].isin(taxa)].set_index(5)[1].to_dict()

In [1]:
def get_uc_read_nr_kraken2(kraken_path, bs_id, krakdb, readpool, mhg, cs, sf, r, bracken_path, level):
    kraken_ucread_nr, kraken_cread_nr = parse_kreport(kraken_path, bs_id, krakdb, readpool, mhg, cs, sf, r)
    return kraken_ucread_nr + kraken_cread_nr - parse_breport(bracken_path, bs_id, krakdb, readpool, mhg, cs, sf, r, level, ['root'])['root']

def get_nonbee_read_nr_kraken2(kraken_path, bs_id, krakdb, readpool, mhg, cs, sf, r):
    "simply sum of kraken2 classified and unclassified here"
    kraken_ucread_nr, kraken_cread_nr = parse_kreport(kraken_path, bs_id, krakdb, readpool, mhg, cs, sf, r)
    return kraken_ucread_nr + kraken_cread_nr